# SDH exp_019 — XGBoost / CatBoost diversity

B10 안전 표현(`contrast=auto`, `drop-exact`)을 고정하고 XGBoost 12개와 CatBoost 12개를 비교합니다. 최종 질문은 신규 모델이 기존 `LR+LGBM` 앙상블 위에서 추가 이득을 내는지입니다.

이 노트북은 test.csv를 읽지 않습니다.

In [ ]:
from pathlib import Path
import importlib.metadata
import json
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

def find_root(start):
    for path in (start, *start.parents):
        if (path / 'data' / 'raw' / 'train.csv').exists():
            return path
    raise FileNotFoundError('data/raw/train.csv가 있는 저장소 루트를 찾지 못했습니다.')

ROOT = find_root(Path.cwd().resolve())
EXP_DIR = ROOT / 'experiments' / 'SDH' / 'exp_019_xgb_catboost_diversity'
RESULT_DIR = EXP_DIR / 'results'
RESULT_DIR.mkdir(exist_ok=True)
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))

import model_experiment as exp

train = pd.read_csv(ROOT / 'data' / 'raw' / 'train.csv')
genes = [column for column in train if column not in ('ID', 'SUBCLASS')]
labels = train['SUBCLASS'].reset_index(drop=True)
classes = np.asarray(sorted(labels.unique()))
assert len(classes) == 26
print('root:', ROOT)
print('train:', train.shape, 'genes:', len(genes), 'classes:', len(classes))
print('xgboost:', importlib.metadata.version('xgboost'))
print('catboost:', importlib.metadata.version('catboost'))

In [ ]:
CASES = exp.case_catalog()
XGB_CASES = {name: case for name, case in CASES.items() if case.family == 'xgb'}
CAT_CASES = {name: case for name, case in CASES.items() if case.family == 'cat'}
SEEDS = (42, 52, 62)
case_table = pd.DataFrame([vars(case) for case in CASES.values()])
print('XGBoost:', len(XGB_CASES), 'CatBoost:', len(CAT_CASES))
display(case_table[['name', 'family', 'view', 'n_estimators', 'depth', 'learning_rate', 'description']])

## 1. Seed 42 B10 안전 표현 준비

각 outer fold에서 원본 fit/validation을 별도 파싱하고, fit에서만 vocabulary·auto contrast·enrichment·hybrid support를 학습합니다. 네 feature view는 모든 모델 case가 공유합니다.

In [ ]:
prepared_by_seed = {}
anchors = {}
started = time.perf_counter()
prepared_by_seed[42] = exp.prepare_seed(train, genes, seed=42, verbose=True)
print(f'prepare seed42: {(time.perf_counter() - started) / 60:.1f} min')

for item in prepared_by_seed[42]:
    audit = item.audit
    assert audit['raw_train_valid_concat'] is False
    assert audit['vocabulary_source'] == 'outer_fold_fit_only'
    assert audit['fixed_exact_event_enabled'] is False
    assert audit['fixed_cancer_pair_enabled'] is False
    assert audit['minimum_full_value'] < 0, 'signed enrichment/contrast가 사라졌습니다.'
print('leakage and signed-feature audit: PASS')
display(pd.DataFrame([{'fold': item.fold, **item.audit['view_feature_counts'], 'min': item.audit['minimum_full_value']} for item in prepared_by_seed[42]]))

In [ ]:
anchors[42] = exp.evaluate_anchor(prepared_by_seed[42], labels, seed=42)
encoded = np.searchsorted(anchors[42].classes, labels)
anchor_metrics_42 = anchors[42].metrics(encoded)
print('safe LR anchor seed42:', anchor_metrics_42)

## 2. Seed 42 — XGBoost 12개

오래 걸리는 셀입니다. case 하나가 실패해도 나머지 case는 계속 실행됩니다.

In [ ]:
screen_results = {}
screen_rows = []
errors = {}

def run_screen(catalog):
    for index, (case_name, case) in enumerate(catalog.items(), start=1):
        started = time.perf_counter()
        try:
            result = exp.evaluate_case(prepared_by_seed[42], labels, case, seed=42)
            row = exp.summary_row(result, anchors[42], labels, case)
            screen_results[case_name] = result
            screen_rows.append(row)
            print(f'[{index:02d}/{len(catalog)}] {case_name}: single={row["oof_macro_f1"]:.6f} blend20={row["blend20_f1"]:.6f} rescue={row["rescue_rate"]:.3f} time={(time.perf_counter()-started)/60:.1f}m')
        except Exception as error:
            errors[case_name] = repr(error)
            print(f'[ERROR] {case_name}: {error!r}')

run_screen(XGB_CASES)
pd.DataFrame(screen_rows).to_csv(RESULT_DIR / 'seed42_screen_partial.csv', index=False)
(RESULT_DIR / 'screen_errors.json').write_text(json.dumps(errors, indent=2, ensure_ascii=False), encoding='utf-8')

## 3. Seed 42 — CatBoost 12개

full depth8과 800-tree case가 가장 오래 걸립니다. XGBoost 결과는 이미 `seed42_screen_partial.csv`에 저장되어 있습니다.

In [ ]:
run_screen(CAT_CASES)
screen_table = pd.DataFrame(screen_rows).sort_values(
    ['blend20_f1', 'rescue_rate', 'oof_macro_f1'], ascending=False
).reset_index(drop=True)
screen_table.to_csv(RESULT_DIR / 'seed42_screen.csv', index=False)
(RESULT_DIR / 'screen_errors.json').write_text(json.dumps(errors, indent=2, ensure_ascii=False), encoding='utf-8')
display(screen_table[['case', 'family', 'view', 'oof_macro_f1', 'blend20_f1', 'blend20_delta', 'disagreement', 'rescue_rate', 'reverse_loss_rate', 'oracle_macro_f1', 'probability_correlation']])

## 4. 구 LR-only 판정 경로 — 실행하지 않음

아래의 기존 후보 선택·3-seed 셀은 결과 호환을 위해 남겨 두었지만 실행하지 않습니다. CatBoost 셀까지 끝난 뒤 노트북 맨 아래 `incremental_recheck.py` 셀로 이동하세요.

In [ ]:
CONFIRM_PER_FAMILY = 3
MIN_PAIRWISE_DISAGREEMENT = 0.01
CONFIRM_CASES = []

for family in ('xgb', 'cat'):
    ranked = screen_table.loc[screen_table['family'] == family, 'case'].tolist()
    selected = []
    for case_name in ranked:
        candidate = screen_results[case_name]
        if not selected or min(np.mean(candidate.prediction_index != screen_results[name].prediction_index) for name in selected) >= MIN_PAIRWISE_DISAGREEMENT:
            selected.append(case_name)
        if len(selected) >= CONFIRM_PER_FAMILY:
            break
    for case_name in ranked:
        if case_name not in selected:
            selected.append(case_name)
        if len(selected) >= CONFIRM_PER_FAMILY:
            break
    CONFIRM_CASES.extend(selected)

print('confirmation cases:')
display(screen_table[screen_table['case'].isin(CONFIRM_CASES)][['case', 'family', 'view', 'oof_macro_f1', 'blend20_f1', 'blend20_delta', 'disagreement', 'rescue_rate']])

## 5. Seeds 52/62 확인

자는 동안 실행하기 좋은 셀입니다. 두 seed의 안전 피처를 각각 다시 만들고 6개 후보를 평가합니다.

In [ ]:
confirmation_results = {(42, name): screen_results[name] for name in CONFIRM_CASES}
confirmation_rows = [screen_table.loc[screen_table['case'] == name].iloc[0].to_dict() for name in CONFIRM_CASES]
confirmation_errors = {}

for seed in (52, 62):
    started = time.perf_counter()
    prepared_by_seed[seed] = exp.prepare_seed(train, genes, seed=seed, verbose=True)
    anchors[seed] = exp.evaluate_anchor(prepared_by_seed[seed], labels, seed=seed)
    for case_name in CONFIRM_CASES:
        try:
            result = exp.evaluate_case(prepared_by_seed[seed], labels, CASES[case_name], seed=seed)
            confirmation_results[(seed, case_name)] = result
            confirmation_rows.append(exp.summary_row(result, anchors[seed], labels, CASES[case_name]))
            print(seed, case_name, confirmation_rows[-1]['oof_macro_f1'], confirmation_rows[-1]['blend20_delta'])
        except Exception as error:
            confirmation_errors[f'{seed}:{case_name}'] = repr(error)
            print('[ERROR]', seed, case_name, repr(error))
    print(f'seed={seed} elapsed: {(time.perf_counter() - started)/60:.1f} min')

confirmation = pd.DataFrame(confirmation_rows)
confirmation.to_csv(RESULT_DIR / 'three_seed_confirmation.csv', index=False)
(RESULT_DIR / 'confirmation_errors.json').write_text(json.dumps(confirmation_errors, indent=2, ensure_ascii=False), encoding='utf-8')
display(confirmation.sort_values(['seed', 'blend20_f1'], ascending=[True, False]))

In [ ]:
summary = (
    confirmation.groupby(['case', 'family', 'view'], as_index=False)
    .agg(
        mean_single_f1=('oof_macro_f1', 'mean'),
        std_single_f1=('oof_macro_f1', 'std'),
        mean_blend20_f1=('blend20_f1', 'mean'),
        mean_blend20_delta=('blend20_delta', 'mean'),
        min_blend20_delta=('blend20_delta', 'min'),
        positive_seeds=('blend20_delta', lambda values: int((values > 0).sum())),
        mean_disagreement=('disagreement', 'mean'),
        mean_rescue=('rescue_rate', 'mean'),
        mean_reverse_loss=('reverse_loss_rate', 'mean'),
        mean_oracle=('oracle_macro_f1', 'mean'),
        mean_probability_correlation=('probability_correlation', 'mean'),
    )
)
summary['promote_to_b10_test'] = (
    (summary['positive_seeds'] == 3)
    & (summary['mean_blend20_delta'] > 0)
    & (summary['mean_disagreement'] >= 0.10)
    & (summary['mean_rescue'] >= 0.10)
)
summary = summary.sort_values(['mean_blend20_f1', 'min_blend20_delta'], ascending=False).reset_index(drop=True)
summary.to_csv(RESULT_DIR / 'three_seed_summary.csv', index=False)
display(summary)
PROMOTED_CASES = summary.loc[summary['promote_to_b10_test'], 'case'].tolist()
print('B10 fold-local blend 검증 승격 후보:', PROMOTED_CASES if PROMOTED_CASES else '없음')

## 6. 승격 후보 OOF 확률 저장

결과 파일은 gitignore 대상입니다. B10과 행·seed·fold를 맞춰 별도 fold-local blend 검증할 수 있도록 저장합니다.

In [ ]:
EXPORT_CASES = PROMOTED_CASES
for seed in SEEDS:
    fold_number = np.zeros(len(train), dtype=np.int8)
    for item in prepared_by_seed[seed]:
        fold_number[item.valid_index] = item.fold
    export_items = [('safe_lr_anchor', anchors[seed])]
    export_items.extend((name, confirmation_results[(seed, name)]) for name in EXPORT_CASES)
    for name, result in export_items:
        frame = pd.DataFrame({'ID': train['ID'], 'SUBCLASS': labels, 'seed': seed, 'fold': fold_number})
        for column, class_name in enumerate(result.classes):
            frame[f'prob__{class_name}'] = result.probability[:, column]
        frame.to_csv(RESULT_DIR / f'oof_{name}_seed{seed}.csv', index=False)
print('saved:', RESULT_DIR)

## 판정

- 기존 LR-only 20% blend는 진단용이며 채택 기준이 아닙니다.
- compact가 full보다 좋으면 tree 모델에는 희소 유전자 열이 방해된 것입니다.
- hybrid가 좋으면 고빈도 유전자 일부와 집계 피처의 조합이 적합합니다.
- 상관과 disagreement가 좋아도 `LR+LGBM` 정답을 더 많이 훼손하면 기각합니다.
- 현재 커널의 기존 결과를 보존한 채 바로 아래 마지막 셀로 strict fold-local 재판정을 실행합니다.

In [ ]:
# 판정 기준 수정: 이미 계산한 screen_results를 재사용합니다. 커널을 재시작하지 마세요.
# 위의 기존 3-seed 확인 셀 대신 이 셀을 실행합니다.
exec((EXP_DIR / 'incremental_recheck.py').read_text(encoding='utf-8'), globals())